# **BBC News Zero-Shot Entity Extraction with GLiNER**

Sources: https://huggingface.co/datasets/csebuetnlp/xlsum | https://www.bbc.com/indonesia

**Table of Contents**
1. Install Libraries         
2. Parameter and Environment Configuration
3. Load GLiNER Model and Tokenizer
4. Text Pre-processing
5. Text Chunking
6. Post-processing Entities
7. Entity Extraction
8. Data Ingestion
9. Quality Evaluation
10. Pipeline Orchestration

**1. Install Libraries**

In [1]:
!pip install -q gliner==0.2.28 pandas transformers pydantic-settings python-dotenv

import json
import logging
import re
import threading
import pandas as pd
import torch

from __future__ import annotations
from pathlib import Path
from typing import Iterable
from gliner import GLiNER
from transformers import AutoTokenizer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.6/245.6 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 31.1 MB/s eta 0:00:00


**2. Parameter and Environment Configuration**

In [2]:
NAMA_FILE = "/content/BBC News.jsonl"
KOLOM_TEKS = "text"
KOLOM_JUDUL = "title"
MAX_BARIS = 500

MODEL_NAME = "urchade/gliner_multi-v2.1"
THRESHOLD = 0.60
MIN_ENTITY_CHARS = 2
CHUNK_SIZE = 384
CHUNK_OVERLAP = 64
MAX_TEXT_CHARS = 50_000
BATCH_SIZE = 16

OUTPUT_FILE = "ner_bbc_news.csv"

LABELS = [
    "nama_kebijakan",
    "instansi_regulator",
    "instrumen_kebijakan",
    "dampak_kebijakan",
    "kelompok_terdampak",
    "alokasi_anggaran",
    "status_regulasi",
    "wilayah_penerapan"
]

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)
logger = logging.getLogger("gliner-ner")
logger.setLevel(logging.ERROR)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if DEVICE == "cuda":
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
    print(f"CUDA version  : {torch.version.cuda}")

**3. Load GLiNER Model and Tokenizer**

In [4]:
model = GLiNER.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
except Exception:
    logger.warning("The tokenizer model could not be loaded directly. Using the DeBERTa fallback tokenizer.")
    tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-large")

INFERENCE_LOCK = threading.RLock()

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

**4. Text Pre-processing**

In [5]:
def normalize_entity(text: str) -> str:
    text = text.strip()
    text = re.sub(r"\s+", " ", text)
    return text.casefold()

def validate_text(text: str) -> str:
    if not isinstance(text, str):
        raise TypeError("The text must be a string.")
    text = text.strip()
    if not text:
        raise ValueError("The text cannot be empty.")
    if len(text) > MAX_TEXT_CHARS:
        raise ValueError(f"Text is too long. Maximum {MAX_TEXT_CHARS:,} characters.")
    return text

**5. Text Chunking**

In [6]:
def make_chunks(text: str) -> list[dict]:
    encoded = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    input_ids = encoded["input_ids"]
    offsets = encoded["offset_mapping"]

    if len(input_ids) <= CHUNK_SIZE:
        return [{"text": text, "char_start": 0, "char_end": len(text)}]

    step = CHUNK_SIZE - CHUNK_OVERLAP
    if step <= 0:
        raise ValueError("CHUNK_OVERLAP must be smaller than CHUNK_SIZE.")

    chunks = []
    start_token = 0

    while start_token < len(input_ids):
        end_token = min(start_token + CHUNK_SIZE, len(input_ids))
        char_start = offsets[start_token][0]
        char_end = offsets[end_token - 1][1]
        chunk_text = text[char_start:char_end]

        if chunk_text.strip():
            chunks.append({
                "text": chunk_text,
                "char_start": char_start,
                "char_end": char_end,
            })

        if end_token >= len(input_ids):
            break
        start_token += step

    return chunks

**6. Post-processing Entities**

In [7]:
def clean_entities(entities: list[dict], chunk_start: int) -> list[dict]:
    results = []
    for entity in entities:
        raw_text = str(entity.get("text", "")).strip()
        label = str(entity.get("label", "")).strip()

        if not raw_text or not label:
            continue
        if len(raw_text) < MIN_ENTITY_CHARS:
            continue

        score = float(entity.get("score", 0.0))
        local_start = entity.get("start")
        local_end = entity.get("end")

        if local_start is None or local_end is None:
            continue

        global_start = chunk_start + int(local_start)
        global_end = chunk_start + int(local_end)

        results.append({
            "text": raw_text,
            "label": label,
            "score": score,
            "start": global_start,
            "end": global_end,
        })
    return results

def deduplicate_entities(entities: list[dict]) -> list[dict]:
    if not entities:
        return []

    sorted_entities = sorted(
        entities,
        key=lambda item: (-item["score"], item["start"], -(item["end"] - item["start"]))
    )

    accepted = []
    for candidate in sorted_entities:
        cand_start, cand_end = candidate["start"], candidate["end"]
        cand_label = candidate["label"].casefold()

        is_overlap = False
        for existing in accepted:
            ex_start, ex_end = existing["start"], existing["end"]
            ex_label = existing["label"].casefold()

            if cand_label == ex_label:
                overlap_len = max(0, min(cand_end, ex_end) - max(cand_start, ex_start))
                if overlap_len > 0:
                    is_overlap = True
                    break

        if not is_overlap:
            accepted.append(candidate)

    accepted.sort(key=lambda item: (item["start"], item["end"]))
    return accepted

**7. Entity Extraction**

In [8]:
def predict_chunks(chunks: list[dict], labels: list[str], threshold: float) -> list[dict]:
    if not chunks:
        return []

    chunk_texts = [chunk["text"] for chunk in chunks]

    with INFERENCE_LOCK:
        with torch.inference_mode():
            predictions = model.inference(chunk_texts, labels, threshold=threshold, batch_size=BATCH_SIZE)

    all_entities = []
    for chunk, entities in zip(chunks, predictions):
        cleaned = clean_entities(entities=entities, chunk_start=chunk["char_start"])
        all_entities.extend(cleaned)

    return all_entities

def extract_entities(text: str, labels: Iterable[str] | None = None, threshold: float | None = None) -> dict:
    text = validate_text(text)
    labels = list(labels if labels is not None else LABELS)

    if not labels:
        raise ValueError("Labels cannot be empty.")

    threshold = threshold if threshold is not None else THRESHOLD
    if not 0.0 <= threshold <= 1.0:
        raise ValueError("The threshold must be between 0 dan 1.")

    chunks = make_chunks(text)
    entities = predict_chunks(chunks=chunks, labels=labels, threshold=threshold)
    entities = deduplicate_entities(entities)

    valid_entities = []
    for entity in entities:
        start, end = int(entity["start"]), int(entity["end"])
        if start < 0 or end > len(text) or start >= end:
            continue

        entity["text"] = text[start:end]
        valid_entities.append(entity)

    valid_entities.sort(key=lambda entity: (entity["start"], entity["end"]))

    return {
        "text": text,
        "entity_count": len(valid_entities),
        "entities": valid_entities,
        "metadata": {
            "model": MODEL_NAME,
            "device": DEVICE,
            "threshold": threshold,
            "chunk_count": len(chunks),
        },
    }

def extract_batch(texts: list[str], labels: Iterable[str] | None = None, threshold: float | None = None) -> list[dict]:
    if not texts:
        return []

    results = []
    for text in texts:
        result = extract_entities(text=text, labels=labels, threshold=threshold)
        results.append(result)
    return results

**8. Data Ingestion**

In [9]:
def load_jsonl(file_path: str) -> pd.DataFrame:
    path = Path(file_path)
    if not path.exists():
        raise FileNotFoundError(f"File not found: {file_path}")

    records = []
    invalid_lines = 0
    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            line = line.strip()
            if not line: continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as exc:
                logger.warning("Malformed JSON on line %s: %s", line_number, exc)
                invalid_lines += 1
                continue

            if not isinstance(record, dict):
                logger.warning("Line %s is not JSON object.", line_number)
                invalid_lines += 1
                continue
            records.append(record)

    df = pd.DataFrame(records)
    logger.info("Successfully loaded %s record.", len(df))
    if invalid_lines: logger.warning("Number of invalid lines: %s", invalid_lines)
    return df

def validate_dataset(df: pd.DataFrame) -> None:
    if KOLOM_TEKS not in df.columns:
        raise ValueError(f"Column '{KOLOM_TEKS}' not found.")

**9. Quality Evaluation**

In [10]:
def generate_enterprise_report(df_pred: pd.DataFrame, total_artikel: int) -> None:
    print("\n" + "=" * 70)
    print("ENTITY REPORT")
    print("=" * 70)

    if df_pred.empty:
        print("No entity detected.")
        print("=" * 70)
        return

    total_entities = len(df_pred)
    unique_entities = df_pred["normalized_entity"].nunique()
    average_score = float(df_pred["score"].mean())
    median_score = float(df_pred["score"].median())

    label_distribution = df_pred["label"].value_counts()

    cross_article = df_pred.groupby(["normalized_entity", "label"])["article_id"].nunique().reset_index(name="article_count")
    repeated_cross_article = cross_article[cross_article["article_count"] > 1]

    very_high = len(df_pred[df_pred["score"] >= 0.80])
    medium = len(df_pred[(df_pred["score"] >= 0.55) & (df_pred["score"] < 0.80)])
    low = len(df_pred[df_pred["score"] < 0.55])

    print(f"Total articles              : {total_artikel}")
    print(f"Total entities               : {total_entities}")
    print(f"Unique entities                : {unique_entities}")
    print(f"Average model score        : {average_score:.4f}")
    print(f"Median model score         : {median_score:.4f}")
    print(f"The entities appear in >1 article: {len(repeated_cross_article)}")
    print("\nScore distribution:")
    print(f"  >= 0.80    : {very_high}")
    print(f"  0.55-0.79  : {medium}")
    print(f"  < 0.55     : {low}")
    print("\nLabel distribution:")
    print(label_distribution)
    print("\nTop 20 entities based on frequency:")
    top_entities = df_pred.groupby(["normalized_entity", "label"]).size().sort_values(ascending=False).head(20)
    print(top_entities)
    print("=" * 70)

**10. Pipeline Orchestration**

In [11]:
def process_dataset() -> pd.DataFrame:

    df_raw = load_jsonl(NAMA_FILE)
    validate_dataset(df_raw)
    df_raw = df_raw.head(MAX_BARIS).copy()
    df_raw[KOLOM_TEKS] = df_raw[KOLOM_TEKS].fillna("").astype(str).str.strip()
    df_raw = df_raw[df_raw[KOLOM_TEKS] != ""].copy()
    df_raw["article_id"] = range(len(df_raw))

    if KOLOM_JUDUL in df_raw.columns:
        df_raw["article_title"] = df_raw[KOLOM_JUDUL].fillna("").astype(str)
    else:
        df_raw["article_title"] = "Artikel_" + df_raw["article_id"].astype(str)

    hasil_ekstraksi = []

    for start in range(0, len(df_raw), BATCH_SIZE):
        end = min(start + BATCH_SIZE, len(df_raw))
        batch_df = df_raw.iloc[start:end]
        batch_texts = batch_df[KOLOM_TEKS].tolist()
        batch_results = extract_batch(texts=batch_texts, labels=LABELS, threshold=THRESHOLD)

        for ((_, article), result) in zip(batch_df.iterrows(), batch_results):
            article_id = int(article["article_id"])
            article_title = str(article["article_title"])
            for entity in result["entities"]:
                normalized = normalize_entity(entity["text"])
                hasil_ekstraksi.append({
                    "article_id": article_id,
                    "article_title": article_title,
                    "entitas_terdeteksi": entity["text"],
                    "normalized_entity": normalized,
                    "label": entity["label"],
                    "score": entity["score"],
                    "start": entity["start"],
                    "end": entity["end"],
                })

    df_hasil = pd.DataFrame(hasil_ekstraksi)
    if df_hasil.empty:
        logger.warning("No entity was found.")
        return df_hasil

    df_hasil = df_hasil.drop_duplicates(subset=["article_id", "normalized_entity", "label", "start", "end"]).reset_index(drop=True)
    df_hasil = df_hasil.sort_values(by=["article_id", "start", "score"], ascending=[True, True, False]).reset_index(drop=True)

    df_hasil.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
    generate_enterprise_report(df_pred=df_hasil, total_artikel=len(df_raw))
    return df_hasil

In [12]:
df_hasil = process_dataset()

print("\n" + "=" * 70)
print("RESULT PREVIEW")
print("=" * 70)
if df_hasil.empty:
    print("No entity.")
else:
    display(df_hasil.head(30))


ENTITY REPORT
Total articles              : 500
Total entities               : 677
Unique entities                : 266
Average model score        : 0.7418
Median model score         : 0.7253
The entities appear in >1 article: 30

Score distribution:
  >= 0.80    : 209
  0.55-0.79  : 468
  < 0.55     : 0

Label distribution:
label
instansi_regulator     401
kelompok_terdampak     137
wilayah_penerapan       95
nama_kebijakan          23
alokasi_anggaran        13
instrumen_kebijakan      6
dampak_kebijakan         2
Name: count, dtype: int64

Top 20 entities based on frequency:
normalized_entity         label             
kpk                       instansi_regulator    50
pssi                      instansi_regulator    36
fifa                      instansi_regulator    30
dpr                       instansi_regulator    22
komnas ham                instansi_regulator    21
pemerintah                instansi_regulator    20
masyarakat                kelompok_terdampak    18
kemenpora   

,article_id,article_title,entitas_terdeteksi,normalized_entity,label,score,start,end
0,1,Mengapa pengungsi Suriah diminta segera mening...,pemerintah Suriah,pemerintah suriah,instansi_regulator,0.684831,2340,2357
1,17,Hanya sekitar 10% penduduk Indonesia punya pen...,ADB,adb,instansi_regulator,0.865133,18,21
2,17,Hanya sekitar 10% penduduk Indonesia punya pen...,ADB,adb,instansi_regulator,0.824330,254,257
3,21,"Tiba di Jakarta, 10 ABK jalani pemeriksaan kes...",Filipina selatan,filipina selatan,wilayah_penerapan,0.620554,1068,1084
4,24,Presiden Filipina umumkan negara dalam keadaan...,Warga,warga,kelompok_terdampak,0.638638,1160,1165
5,26,"Jokowi tawarkan pencabutan sanksi, PSSI tetap ...",Kemenpora,kemenpora,instansi_regulator,0.909646,0,9
6,26,"Jokowi tawarkan pencabutan sanksi, PSSI tetap ...",PSSI,pssi,instansi_regulator,0.888278,21,25
7,26,"Jokowi tawarkan pencabutan sanksi, PSSI tetap ...",PSSI,pssi,instansi_regulator,0.876941,115,119
8,26,"Jokowi tawarkan pencabutan sanksi, PSSI tetap ...",FIFA,fifa,instansi_regulator,0.814368,294,298
9,26,"Jokowi tawarkan pencabutan sanksi, PSSI tetap ...",PSSI,pssi,instansi_regulator,0.788219,356,360
